# 0.6 Adaptation Frequency Prep

This prep notebook builds a basin-scale frequency adaptation scenario and
writes a scenario basin risk CSV that can be consumed by the simulation
notebooks.


In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import numpy as np
import pandas as pd

from sovereign.flood import (
    apply_basin_frequency_shift,
    build_uniform_frequency_shift_table,
)


In [2]:
# USER CONFIG
model = "wri"
scenario_name = "nbs_shift_150pct"
shift_factor = 1.50
degrade_protection = True
n_target_basins = 10


In [3]:
# Paths and baseline inputs
root = Path.cwd().parent
risk_basin_path = root / "outputs" / "flood" / "risk" / "basins" / f"risk_basins_m-{model}.csv"

scenario_root = root / "outputs" / "flood" / "adaptation" / "frequency" / scenario_name
scenario_basin_dir = scenario_root / "basins"
scenario_basin_dir.mkdir(parents=True, exist_ok=True)

risk_data = pd.read_csv(risk_basin_path)
risk_data = risk_data.iloc[:, 1:] if "Unnamed: 0" in risk_data.columns[0] else risk_data
risk_data["AEP"] = 1 / risk_data["RP"]
risk_data["Pr_L_AEP"] = np.where(risk_data["Pr_L"] == 0, 0, 1 / risk_data["Pr_L"])
risk_data.reset_index(drop=True, inplace=True)

# Example target basins: first N unique basins from the risk table
target_basins = sorted(risk_data["HB_L6"].unique())[:n_target_basins]
target_basins[:10]


[1060999120.0,
 1061016240.0,
 1061022540.0,
 1061029030.0,
 1061033480.0,
 1061033490.0,
 1061041420.0,
 1061041490.0,
 1061051360.0,
 1061051510.0]

In [4]:
# Build frequency shift table
frequency_shift_df = build_uniform_frequency_shift_table(
    return_periods=risk_data["RP"].unique(),
    shift_factor=shift_factor,
)
frequency_shift_df


,RP,RP_future
0,5.0,7.5
1,10.0,15.0
2,25.0,37.5
3,50.0,75.0
4,100.0,150.0
5,250.0,375.0
6,500.0,750.0
7,1000.0,1500.0


In [5]:
# Build scenario basin dataframe
scenario_risk_df = apply_basin_frequency_shift(
    risk_df=risk_data,
    frequency_shift_df=frequency_shift_df,
    basin_ids=target_basins,
    degrade_protection=degrade_protection,
)

scenario_basin_path = scenario_basin_dir / f"risk_basins_m-{model}.csv"
scenario_risk_df.to_csv(scenario_basin_path, index=False)

print("Scenario basin CSV written to:")
print(scenario_basin_path)
scenario_risk_df.head()


Scenario basin CSV written to:
E:\Projects\sovereign-risk-uga\outputs\flood\adaptation\frequency\nbs_shift_150pct\basins\risk_basins_m-wri.csv


,FID,GID_1,NAME,HB_L6,Pr_L,damages,adapted_damages,RP,Sector,AEP,Pr_L_AEP,component_type
0,0,UGA.3_1,Arua,1.061054e+09,2.0,0.000000,0.000000,5.0,Public,0.200000,0.500000,frequency_shifted
1,1,UGA.47_1,Nebbi,1.061054e+09,2.0,0.000000,0.000000,5.0,Public,0.200000,0.500000,frequency_shifted
2,2,UGA.27_1,Kitgum,1.060999e+09,7.5,0.000000,0.000000,7.5,Public,0.133333,0.133333,frequency_shifted
3,3,UGA.41_1,Moyo,1.061033e+09,7.5,0.000000,0.000000,7.5,Public,0.133333,0.133333,frequency_shifted
4,4,UGA.3_1,Arua,1.061033e+09,7.5,6160.324707,6160.324707,7.5,Public,0.133333,0.133333,frequency_shifted


## Notes

To customize this scenario, change:

- `scenario_name`
- `shift_factor`
- `degrade_protection`
- `target_basins`

If you want a non-uniform shift, replace `build_uniform_frequency_shift_table(...)`
with your own dataframe containing either:

- `RP` and `RP_future`
- or `AEP` and `AEP_future`
